<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 El Examen Sorpresa, la Moneda y el Pronosticador Honesto 🌦️
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 07
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/07%20-%20Seleccion%20de%20Modelos%20y%20Optimizacion/Para%20Dummies/00_Evaluacion_Seleccion_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 00 (Evaluación y selección de modelos)](../00_Evaluacion_y_Seleccion_de_Modelos.ipynb). Ya sabes que un modelo se «califica» con números como el *accuracy*. Aquí aprendemos a **creerle (o no) a esos números**: ¿el modelo aprendió de verdad o solo memorizó?, ¿una diferencia de un punto entre dos modelos es real o es suerte?, ¿puedo fiarme cuando el modelo dice «80 % seguro»? Solo usamos *NumPy* y gráficas.

Al terminar podrás explicar:
1. Por qué existe un **examen final sellado** que no se puede mirar antes de tiempo.
2. Por qué una diferencia pequeña entre dos modelos **puede ser pura suerte** y qué es un **intervalo de confianza** (con una moneda).
3. Qué significa que un modelo sea **honesto con sus probabilidades** (el pronosticador del tiempo).
4. Que la mejor medida **depende de qué error duele más** (la alarma de incendios).

> 🔗 Si luego quieres ver la versión con más detalle: [cuaderno estándar](../00_Evaluacion_y_Seleccion_de_Modelos.ipynb). Antes de esto, el [Módulo 03 (métricas)](../../03%20-%20Metricas%20y%20Ajuste%20de%20Hiperparametros/README.md).

---
## 1. El estudiante memorión y el examen final sellado 📝

Imagina tres momentos de un curso:

| En el curso | En machine learning | Para qué sirve |
|---|---|---|
| **Ejercicios de práctica** (con respuestas) | Datos de **entrenamiento** | Aprender |
| **Simulacro** (para comparar métodos de estudio) | Datos de **validación** | Elegir *cómo* estudiar |
| **Examen final** (con llave: se abre **una sola vez**) | Datos de **prueba** | Saber cuánto sabes **de verdad** |

Si alguien ve las preguntas del examen final antes, su nota **ya no mide lo que sabe**. Vamos a verlo con un **estudiante memorión**: le damos 200 preguntas de verdadero/falso **cuyas respuestas se decidieron lanzando una moneda** (no hay nada que aprender). El memorión se aprende todas las respuestas de la práctica de memoria. ¿Cómo le va en la práctica y en un examen con preguntas nuevas?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)
# 200 preguntas, cada una descrita por 10 numeritos; la respuesta (V/F) se decide con una MONEDA: no hay nada que aprender
preguntas_practica = rng.normal(size=(200, 10)); respuestas_practica = rng.integers(0, 2, 200)
preguntas_examen = rng.normal(size=(200, 10));   respuestas_examen = rng.integers(0, 2, 200)

def memorion(preguntas, memoria_X, memoria_y):
    """Responde copiando la respuesta de la pregunta MAS PARECIDA que memorizó (el vecino más cercano)."""
    d = ((preguntas[:, None, :] - memoria_X[None, :, :]) ** 2).sum(axis=2)
    return memoria_y[d.argmin(axis=1)]

nota_practica = np.mean(memorion(preguntas_practica, preguntas_practica, respuestas_practica) == respuestas_practica)
nota_examen = np.mean(memorion(preguntas_examen, preguntas_practica, respuestas_practica) == respuestas_examen)
print(f"Nota del memorión en la PRÁCTICA (preguntas que ya vio): {nota_practica:.0%}")
print(f"Nota del memorión en el EXAMEN con preguntas nuevas:     {nota_examen:.0%}  (lo esperado al azar es 50 %)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(["Práctica\n(ya vista)", "Examen final\n(preguntas nuevas)"], [nota_practica, nota_examen], color=["#dc2626", "#16a34a"])
ax.axhline(0.5, color="black", ls="--", label="Adivinar con una moneda"); ax.set_ylim(0, 1.1); ax.set_ylabel("Fracción de respuestas correctas"); ax.legend()
ax.set_title("Memorizar no es aprender: el examen sellado lo revela", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
assert nota_practica == 1.0 and abs(nota_examen - 0.5) < 0.1

---
**Lección:** en la práctica el memorión saca **100 %** y en el examen real **alrededor de 50 %**, como lanzar una moneda. Un modelo evaluado con los mismos datos con los que aprendió **siempre** se ve genial. Por eso el protocolo es: **separar el examen final al principio, sellarlo, y abrirlo una sola vez al final**. Y antes de presumir cualquier nota, hay que compararla con una **línea base** (por ejemplo, «responder siempre lo más común»).

## 2. ¿Esa diferencia es real o es suerte? 🪙

Dos modelos hacen el examen de **100 preguntas**: el modelo A acierta 91 y el modelo B acierta 90. ¿A es mejor? Piensa en una **moneda justa**: lanzada 100 veces no siempre da 50 caras; a veces 46, a veces 55. Con un examen de 100 preguntas pasa lo mismo: **otro examen habría dado otra nota**.

Para medir cuánto «tiembla» una nota usamos un truco: **volver a sortear tus propios aciertos**. Tomamos los 100 resultados (91 aciertos y 9 fallos), los «barajamos con repetición» muchas veces y vemos entre qué valores cae la nota. Eso se llama **intervalo de confianza** (aquí del 95 %: el rango donde caería la nota la gran mayoría de las veces).

In [ ]:
rng = np.random.default_rng(2)

def intervalo(aciertos, total, repeticiones=5000):
    """Re-sortea los resultados (1 = acierto, 0 = fallo) y devuelve el rango que cubre al 95 % de las notas."""
    resultados = np.r_[np.ones(aciertos), np.zeros(total - aciertos)]
    notas = rng.choice(resultados, size=(repeticiones, total), replace=True).mean(axis=1)
    return np.percentile(notas, [2.5, 97.5]), notas

for aciertos, total in [(91, 100), (9100, 10000)]:
    (bajo, alto), notas = intervalo(aciertos, total)
    print(f"Examen de {total:>6} preguntas con {aciertos / total:.0%} de aciertos -> la nota 'verdadera' está entre {bajo:.1%} y {alto:.1%}  (ancho = {100 * (alto - bajo):.1f} puntos)")

fig, ax = plt.subplots(figsize=(9, 4))
_, n100 = intervalo(91, 100); _, n10000 = intervalo(9100, 10000)
ax.hist(n100, bins=25, color="#f59e0b", alpha=0.8, density=True, label="Examen de 100 preguntas")
ax.hist(n10000, bins=25, color="#2563eb", alpha=0.8, density=True, label="Examen de 10 000 preguntas")
ax.axvline(0.90, color="black", ls="--", label="La nota del modelo B (90 %)")
ax.set_xlabel("Nota que podría salir con otro examen"); ax.set_ylabel("Densidad"); ax.legend(fontsize=8)
ax.set_title("Con pocas preguntas la nota 'tiembla' mucho: 91 % vs 90 % no se distingue", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
(b100, a100), _ = intervalo(91, 100); (b10k, a10k), _ = intervalo(9100, 10000)
assert b100 < 0.90 < a100                      # con 100 preguntas, el 90 % de B cabe dentro del rango de A: no hay diferencia clara
assert (a10k - b10k) < (a100 - b100) / 5       # con 100 veces mas preguntas el rango se estrecha unas 10 veces

---
**Lección:** con 100 preguntas, la nota del modelo A (91 %) tiene un «margen de error» de unos **±6 puntos**, y el 90 % de B cae **dentro** de ese margen: **no podemos decir que A sea mejor**. Con 10 000 preguntas el margen baja a menos de ±1 punto. Reglas: **(1)** nunca reportes una nota sin su margen; **(2)** para ver diferencias pequeñas hacen falta **muchas** preguntas; **(3)** si no hay diferencia clara, **elige el modelo más sencillo y barato**.

## 3. El pronosticador honesto 🌦️

Un pronosticador dice cada día «hay **70 %** de lluvia». Si es **honesto** (los expertos dicen *calibrado*), entre todos los días en que dijo 70 %, **llovió en 7 de cada 10**. Hay dos pronosticadores que **ordenan igual** los días (ambos saben cuáles son más lluviosos), pero uno exagera:

- **El sabio** dice la probabilidad tal cual.
- **El exagerado** empuja todo hacia los extremos: dice «95 %» o «5 %» casi siempre.

Los dos tienen la misma «puntería para ordenar», pero **no** puedes usar las cifras del exagerado para decidir si llevar paraguas o cuánto cobrar de seguro. Lo comprobamos agrupando los días por lo que dijo cada uno:

In [ ]:
rng = np.random.default_rng(3)
dias = 5000
real = rng.uniform(0.05, 0.95, dias)                      # la probabilidad REAL de lluvia de cada día
llovio = rng.random(dias) < real                          # lo que de verdad pasó
sabio = real                                              # dice la verdad
exagerado = 1 / (1 + np.exp(-4 * np.log(real / (1 - real))))   # misma ordenación, pero empujado hacia 0 % y 100 %

def tabla(dijo, etiqueta):
    print(f"\n{etiqueta}: lo que dijo  ->  lo que pasó de verdad")
    bordes = np.linspace(0, 1, 6)
    for a, b in zip(bordes[:-1], bordes[1:]):
        m = (dijo >= a) & (dijo < b + (1e-9 if b == 1 else 0))
        if m.sum() > 0: print(f"   dijo {a:.0%}-{b:.0%} ({m.sum():>4} días) -> llovió el {llovio[m].mean():.0%}")
tabla(sabio, "El SABIO"); tabla(exagerado, "El EXAGERADO")

def error_cuadratico(p): return np.mean((p - llovio) ** 2)        # 'castigo' por cada pronóstico: menor = mejor
print(f"\nError promedio de las probabilidades (menor es mejor): sabio = {error_cuadratico(sabio):.3f} | exagerado = {error_cuadratico(exagerado):.3f}")

fig, ax = plt.subplots(figsize=(6.2, 5))
ax.plot([0, 1], [0, 1], "k--", label="Honestidad perfecta")
for dijo, nom, col in [(sabio, "Sabio", "#16a34a"), (exagerado, "Exagerado", "#dc2626")]:
    bordes = np.linspace(0, 1, 11); ids = np.clip(np.digitize(dijo, bordes) - 1, 0, 9)
    xs = [dijo[ids == k].mean() for k in range(10) if (ids == k).sum() > 20]; ys = [llovio[ids == k].mean() for k in range(10) if (ids == k).sum() > 20]
    ax.plot(xs, ys, "o-", color=col, label=nom)
ax.set_xlabel("Lo que dijo el pronosticador"); ax.set_ylabel("Lo que pasó realmente"); ax.legend()
ax.set_title("Diagrama de honestidad (calibración)", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
assert error_cuadratico(sabio) < error_cuadratico(exagerado)

---
**Lección:** el sabio es honesto (cuando dice «entre 0 y 20 %» llueve ~12 %, y cuando dice «entre 80 y 100 %» llueve ~87 %); el exagerado **suena muy seguro pero se equivoca con las cifras**: cuando dice «entre 0 y 20 %» llueve en realidad el ~24 %, y cuando dice «entre 80 y 100 %» llueve el ~78 %. Aunque ambos ordenen igual los días, el exagerado se equivoca más con sus cifras. En machine learning, algunos modelos (como los *Naive Bayes*) son «exagerados» y se **corrigen** con una **calibración**: un pequeño ajuste que traduce sus cifras a probabilidades honestas, sin cambiar el orden.

## 4. La alarma de incendios: elige la medida según lo que duele 🔥

Un detector de humo puede equivocarse de dos maneras: **falsa alarma** (suena sin fuego: molesta) o **no sonar cuando hay fuego** (¡catástrofe!). ¿Cuál es «la mejor» alarma? Depende de **cuánto duele cada error**. Supón que una falsa alarma «duele» **1** punto y un incendio que no suena duele **20**. Probamos distintos «niveles de sensibilidad» y sumamos el dolor:

In [ ]:
rng = np.random.default_rng(4)
n = 5000
hay_fuego = rng.random(n) < 0.05                                # 5 % de las veces hay fuego
nivel_humo = np.where(hay_fuego, rng.normal(2.0, 1.0, n), rng.normal(0.0, 1.0, n))   # el detector mide "cuánto humo" (con ruido)

sensibilidades = np.linspace(-1, 4, 101)                        # la alarma suena si el humo supera este valor
def dolor(umbral, costo_falsa, costo_no_sonar):
    suena = nivel_humo > umbral
    falsas, perdidos = np.sum(suena & ~hay_fuego), np.sum(~suena & hay_fuego)
    return (costo_falsa * falsas + costo_no_sonar * perdidos) / n

fig, ax = plt.subplots(figsize=(8.5, 4.4))
for (cf, cn), col in [((1, 1), "#2563eb"), ((1, 20), "#dc2626"), ((20, 1), "#16a34a")]:
    d = np.array([dolor(u, cf, cn) for u in sensibilidades]); u_opt = sensibilidades[d.argmin()]
    ax.plot(sensibilidades, d, color=col, lw=2, label=f"falsa alarma = {cf}, no sonar = {cn}  ->  mejor nivel = {u_opt:.1f}")
    ax.scatter([u_opt], [d.min()], color=col, s=70, zorder=5)
ax.set_xlabel("Nivel de humo a partir del cual suena la alarma (más alto = menos sensible)"); ax.set_ylabel("Dolor promedio por detección")
ax.set_title("Cada negocio tiene su alarma ideal: depende del costo de los errores", fontweight="bold", fontsize=10); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
u_sensible = sensibilidades[np.argmin([dolor(u, 1, 20) for u in sensibilidades])]
u_cauto = sensibilidades[np.argmin([dolor(u, 20, 1) for u in sensibilidades])]
assert u_sensible < u_cauto                                     # si no sonar duele mucho, hay que ser MAS sensible (umbral mas bajo)
print(f"Si no sonar duele 20 veces más: conviene una alarma MUY sensible (nivel {u_sensible:.1f}). Si la falsa alarma duele 20 veces más: conviene una alarma cautelosa (nivel {u_cauto:.1f}).")

---
##### 🛠️ Práctica 1: ¿Cuántas preguntas hacen falta para distinguir 91 % de 90 %?

Antes te dijimos que con **100 preguntas** no se distingue un 91 % de un 90 %. ¿Cuántas preguntas harían falta? Usa la función `intervalo()` de la sección 2 (re-sortea los aciertos y devuelve el rango del 95 %). Para exámenes de 100, 400, 1600 y 6400 preguntas, todos con **91 %** de aciertos:

1. Para cada tamaño calcula los aciertos (`round(0.91 * total)`) y llama a `intervalo(aciertos, total, repeticiones=1000)`.
2. Guarda en dos diccionarios, `anchos` y `bajos`, el ancho del rango (`alto - bajo`) y su extremo inferior para cada tamaño.
3. Busca el **primer** tamaño en el que el extremo inferior ya supera 0.90 (ahí el 90 % de B queda fuera del rango de A) y guárdalo en `total_claro`.
4. Comprueba con `assert` que al **cuadruplicar** las preguntas el ancho baja (más o menos a la mitad) y que `total_claro` es mayor que 1600.

In [ ]:
# Escribe tu código aquí

# totales = [100, 400, 1600, 6400]
# anchos, bajos = {}, {}
# for total in totales:
#     aciertos = round(0.91 * total)
#     (bajo, alto), _ = intervalo(aciertos, total, repeticiones=1000)
#     anchos[total] = ...
#     bajos[total] = ...
# total_claro = next(t for t in totales if bajos[t] > ...)


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
totales = [100, 400, 1600, 6400]
anchos, bajos = {}, {}
for total in totales:
    aciertos = round(0.91 * total)
    (bajo, alto), _ = intervalo(aciertos, total, repeticiones=1000)    # re-sortear los aciertos 1000 veces
    anchos[total] = alto - bajo                                         # cuanto 'tiembla' la nota
    bajos[total] = bajo                                                 # extremo de abajo del rango

for t in totales:
    print(f"{t:>5} preguntas: rango del 95 % desde {bajos[t]:.1%} (ancho = {100 * anchos[t]:.1f} puntos)")

total_claro = next(t for t in totales if bajos[t] > 0.90)             # primer examen donde el 90 % de B queda FUERA del rango
print(f"Primer examen donde 91 % se distingue de 90 %: {total_claro} preguntas")

assert all(anchos[b] < anchos[a] for a, b in zip(totales[:-1], totales[1:]))   # al cuadruplicar, el ancho baja
assert anchos[100] / anchos[400] > 1.5                                          # y baja a casi la mitad
assert total_claro > 1600
```
</details>

---


---
## Resumen en una frase 🎯

> **Un modelo se evalúa con un examen final sellado que se abre una sola vez; toda nota viene con su margen de error (una diferencia pequeña puede ser suerte); las probabilidades deben ser honestas (calibradas); y la «mejor» medida depende de cuánto duele cada tipo de error.**

### Lo que aprendiste hoy:
- ✅ **Memorizar no es aprender**: el memorión sacó 100 % en la práctica y ≈ 50 % en el examen nuevo. Separa el examen final y ábrelo **una vez**.
- ✅ **Toda nota tiembla** (como una moneda): 91 % vs 90 % en 100 preguntas **no** es una diferencia real; con 10 000 preguntas el margen baja a menos de ±1 punto.
- ✅ **Honestidad de las probabilidades**: dos pronosticadores pueden ordenar igual y aun así uno exagera; se corrige con calibración.
- ✅ **La alarma ideal depende del dolor de cada error**: elige la medida **después** de preguntar qué error cuesta más.

> 🎓 **Siguiente paso:** explora el [cuaderno estándar](../00_Evaluacion_y_Seleccion_de_Modelos.ipynb) (pruebas estadísticas para comparar modelos y calibración real) y continúa con [los exámenes de práctica justos (validación cruzada)](01_Validacion_Cruzada_Avanzada_Dummies.ipynb).

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿por qué un estudiante que **memoriza** las respuestas de la práctica saca 100 % ahí y ≈ 50 % en el examen real? Pon un ejemplo de tu vida (aprender un idioma, una receta).
2. Un amigo te dice: «mi modelo saca 92 % y el tuyo 91 %: ¡el mío es mejor!». ¿Qué le preguntarías antes de creerle? ¿Cuántas preguntas tenía el examen?
3. Un servicio de salud quiere un detector de una enfermedad grave pero tratable. ¿Preferirías una alarma sensible o cautelosa? ¿Por qué?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>